# A/B Test Analysis for an E-Commerce Website
## Comparing Two Pricing Strategies Using Hypothesis Testing

**Purpose:** Compare conversion outcomes for the control (old page) and treatment (new page) groups, then examine conversion across countries and country–strategy combinations.

This notebook follows the structure and tests in the supplied reference report. Results are calculated from the datasets at runtime; the report's values are not hardcoded.

**Significance level:** α = 0.05.

> **Data access note:** The GitHub raw URLs are used below. If your current environment cannot access GitHub, download both CSV files and set `LOCAL_DATA_DIR` in the configuration cell.

## 1. Import libraries

We use pandas and NumPy for data handling, SciPy for hypothesis tests, and Matplotlib for simple plots.

In [ ]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from IPython.display import display

ALPHA = 0.05
AB_URL = "https://raw.githubusercontent.com/aamir-gk2539-byte/DSML-Training/main/ab_data.csv"
COUNTRIES_URL = "https://raw.githubusercontent.com/aamir-gk2539-byte/DSML-Training/main/countries.csv"

# If GitHub access fails, place both CSV files in a folder and enter its path here.
# Example: LOCAL_DATA_DIR = Path(r"C:/Users/YourName/Downloads")
LOCAL_DATA_DIR = None

## 2. Load the datasets

The A/B dataset should contain each user's experimental group, landing page, and conversion outcome. The countries dataset maps user IDs to countries.

The loader tries the GitHub raw URL first. If it fails, it tries the local file (when `LOCAL_DATA_DIR` is set). It raises a clear error rather than silently creating or inventing data.

In [ ]:
def load_csv(url, filename, local_dir=None):
    """Load a CSV from a URL, with an optional local-file fallback."""
    errors = []
    try:
        df = pd.read_csv(url)
        print(f"Loaded {filename} from GitHub: {url}")
        return df
    except Exception as exc:
        errors.append(f"GitHub download failed: {type(exc).__name__}: {exc}")

    if local_dir is not None:
        local_path = Path(local_dir) / filename
        try:
            df = pd.read_csv(local_path)
            print(f"Loaded {filename} from local file: {local_path}")
            return df
        except Exception as exc:
            errors.append(f"Local-file load failed ({local_path}): {type(exc).__name__}: {exc}")

    details = "\n".join(errors)
    raise RuntimeError(
        f"Could not load {filename}.\n{details}\n"
        "Download the CSV from the project GitHub repository, place it in a folder, "
        "set LOCAL_DATA_DIR to that folder, and rerun this cell."
    )

try:
    ab_data = load_csv(AB_URL, "ab_data.csv", LOCAL_DATA_DIR)
    countries = load_csv(COUNTRIES_URL, "countries.csv", LOCAL_DATA_DIR)
except RuntimeError as exc:
    print(exc)
    raise

In [ ]:
def inspect_dataset(df, name):
    print(f"\\n--- {name} ---")
    print("Shape:", df.shape)
    print("Columns:", df.columns.tolist())
    print("\\nFirst five rows:")
    display(df.head())
    print("\\nData types:")
    display(df.dtypes.rename("dtype").to_frame())
    print("\\nMissing values per column:")
    display(df.isna().sum().rename("missing_values").to_frame())
    if "user_id" in df.columns:
        print("Unique user IDs:", df["user_id"].nunique())

inspect_dataset(ab_data, "A/B testing dataset")
inspect_dataset(countries, "Countries dataset")

## 3. Understand and validate the data

- **Control** is expected to mean the visitor was shown the existing/old page.
- **Treatment** is expected to mean the visitor was shown the new page.
- **`converted = 1`** means a conversion occurred; **`converted = 0`** means it did not.
- Country information lets us check whether conversion patterns vary geographically.

The code checks for the expected columns and handles harmless capitalization/spacing differences in column names. It does not guess a replacement for a genuinely missing required field.

In [ ]:
def normalize_column_names(df):
    df = df.copy()
    df.columns = [str(c).strip().lower().replace(" ", "_") for c in df.columns]
    return df

ab_data = normalize_column_names(ab_data)
countries = normalize_column_names(countries)

required_ab = {"user_id", "group", "landing_page", "converted"}
required_countries = {"user_id", "country"}
missing_ab = required_ab - set(ab_data.columns)
missing_countries = required_countries - set(countries.columns)

if missing_ab:
    raise ValueError(f"A/B dataset is missing required columns: {sorted(missing_ab)}. Found: {ab_data.columns.tolist()}")
if missing_countries:
    raise ValueError(f"Countries dataset is missing required columns: {sorted(missing_countries)}. Found: {countries.columns.tolist()}")

# Parse timestamps if present, to allow retaining the first recorded visit reliably.
if "timestamp" in ab_data.columns:
    ab_data["timestamp"] = pd.to_datetime(ab_data["timestamp"], errors="coerce")

print("A/B dataset group values:", sorted(ab_data["group"].dropna().astype(str).unique().tolist()))
print("Landing page values:", sorted(ab_data["landing_page"].dropna().astype(str).unique().tolist()))
print("Conversion values:", sorted(ab_data["converted"].dropna().unique().tolist()))
print("Country values:", sorted(countries["country"].dropna().astype(str).unique().tolist()))

## 4. Data cleaning

The reference report describes two main steps:

1. Remove rows where group and landing page do not match: control → old page; treatment → new page.
2. Remove duplicate user IDs, retaining the first recorded visit.
3. Merge the cleaned A/B data with the country lookup by `user_id`.

Before cleaning, we count mismatches, duplicate IDs, and missing values. We then check merge cardinality to avoid accidentally multiplying rows. Country values and page/group labels are compared case-insensitively after trimming whitespace.

In [ ]:
# Work on copies so the raw data remains available for inspection.
ab_clean = ab_data.copy()
countries_clean = countries.copy()

# Standardize labels without changing the underlying meaning.
ab_clean["group"] = ab_clean["group"].astype("string").str.strip().str.lower()
ab_clean["landing_page"] = ab_clean["landing_page"].astype("string").str.strip().str.lower()
ab_clean["converted"] = pd.to_numeric(ab_clean["converted"], errors="coerce")

print("BEFORE CLEANING")
print("Rows:", len(ab_clean))
print("Duplicate user_id rows beyond first:", int(ab_clean["user_id"].duplicated().sum()))
print("Rows with missing values:", int(ab_clean.isna().any(axis=1).sum()))
print("Missing values by column:")
display(ab_clean.isna().sum().rename("missing_values").to_frame())

# Recognize the expected labels, allowing small text-format differences.
control_labels = {"control"}
treatment_labels = {"treatment"}
old_page_labels = {"old_page", "old page", "old"}
new_page_labels = {"new_page", "new page", "new"}

is_control = ab_clean["group"].isin(control_labels)
is_treatment = ab_clean["group"].isin(treatment_labels)
is_old = ab_clean["landing_page"].isin(old_page_labels)
is_new = ab_clean["landing_page"].isin(new_page_labels)

matches = (is_control & is_old) | (is_treatment & is_new)
mismatch_mask = ~matches
print("Group/page mismatches (including unrecognized labels):", int(mismatch_mask.sum()))

# Keep only correctly assigned group/page combinations.
ab_clean = ab_clean.loc[matches].copy()

# Sort by timestamp where available, then retain the first record for each user.
if "timestamp" in ab_clean.columns:
    ab_clean = ab_clean.sort_values("timestamp", kind="stable", na_position="last")
ab_clean = ab_clean.drop_duplicates(subset="user_id", keep="first").copy()

# Validate conversion data; rows without a usable binary outcome cannot enter these tests.
before_valid_outcome = len(ab_clean)
ab_clean = ab_clean.loc[ab_clean["converted"].isin([0, 1])].copy()
print("Rows removed for missing/non-binary converted value:", before_valid_outcome - len(ab_clean))

# Clean country lookup and guard against duplicate keys before merge.
countries_clean["country"] = countries_clean["country"].astype("string").str.strip()
duplicate_country_keys = int(countries_clean["user_id"].duplicated().sum())
print("Duplicate user_id keys in countries dataset:", duplicate_country_keys)
if duplicate_country_keys:
    # The lookup should have one country per user. Keep the first record to prevent row multiplication.
    countries_clean = countries_clean.drop_duplicates(subset="user_id", keep="first").copy()

rows_before_merge = len(ab_clean)
merged = ab_clean.merge(
    countries_clean[["user_id", "country"]],
    on="user_id",
    how="left",
    validate="one_to_one",
    indicator=True
)
print("Rows before merge:", rows_before_merge)
print("Rows after merge:", len(merged))
print("Rows without country match:", int((merged["_merge"] != "both").sum()))
assert len(merged) == rows_before_merge, "Merge unexpectedly changed the number of A/B rows."

# Country is required for country-specific analyses. Keep matched records for all analyses
# so the overall and geographic tests use the same analyzed sample.
merged = merged.loc[merged["_merge"] == "both"].drop(columns="_merge").copy()

print("\\nAFTER CLEANING AND MERGE")
print("Rows:", len(merged))
print("Unique users:", merged["user_id"].nunique())
print("Group sizes:")
display(merged["group"].value_counts().rename_axis("group").to_frame("users"))
print("Country values:", sorted(merged["country"].dropna().astype(str).unique().tolist()))

## 5. Descriptive statistics

Because `converted` is binary (0/1), its mean equals the conversion rate. The difference below is **treatment minus control**, expressed in percentage points (pp).

In [ ]:
group_summary = (
    merged.groupby("group", observed=True)
    .agg(users=("user_id", "nunique"),
         conversions=("converted", "sum"),
         conversion_rate=("converted", "mean"))
    .reindex(["control", "treatment"])
)
group_summary["non_conversions"] = group_summary["users"] - group_summary["conversions"]
group_summary["conversion_rate_percent"] = group_summary["conversion_rate"] * 100
display(group_summary)

if {"control", "treatment"}.issubset(group_summary.index) and group_summary["conversion_rate"].notna().all():
    rate_diff_pp = (group_summary.loc["treatment", "conversion_rate"] -
                    group_summary.loc["control", "conversion_rate"]) * 100
    print(f"Treatment − control conversion-rate difference: {rate_diff_pp:.4f} percentage points")

ax = (group_summary["conversion_rate"] * 100).plot(kind="bar", rot=0)
ax.set_title("Conversion rate by experimental group")
ax.set_xlabel("Experimental group")
ax.set_ylabel("Conversion rate (%)")
plt.tight_layout()
plt.show()

## 6. Hypothesis Test 1 — Welch independent-samples t-test

**H₀:** Mean conversion outcomes are equal for control and treatment.  
**H₁:** Mean conversion outcomes differ between control and treatment.

Welch's t-test does not assume equal variances. Since conversion is binary, the group mean is the conversion rate. A two-proportion test or chi-square test is often more natural for binary outcomes, but the t-test is included to match the reference report.

We also calculate a 95% Welch confidence interval for the difference **treatment − control**.

In [ ]:
control = merged.loc[merged["group"] == "control", "converted"].astype(float).to_numpy()
treatment = merged.loc[merged["group"] == "treatment", "converted"].astype(float).to_numpy()

if len(control) < 2 or len(treatment) < 2:
    raise ValueError("At least two valid observations per group are required for the t-test.")

t_stat, t_pvalue = stats.ttest_ind(treatment, control, equal_var=False, nan_policy="omit")
mean_treatment = float(np.mean(treatment))
mean_control = float(np.mean(control))
difference = mean_treatment - mean_control

# Welch standard error and Satterthwaite degrees of freedom.
var_t = np.var(treatment, ddof=1)
var_c = np.var(control, ddof=1)
se_diff = np.sqrt(var_t / len(treatment) + var_c / len(control))
df_num = (var_t / len(treatment) + var_c / len(control)) ** 2
df_den = ((var_t / len(treatment)) ** 2 / (len(treatment) - 1) +
          (var_c / len(control)) ** 2 / (len(control) - 1))
welch_df = df_num / df_den
critical_t = stats.t.ppf(1 - ALPHA / 2, welch_df)
ci_low, ci_high = difference - critical_t * se_diff, difference + critical_t * se_diff

print(f"Control mean conversion:   {mean_control:.6f} ({mean_control*100:.3f}%)")
print(f"Treatment mean conversion: {mean_treatment:.6f} ({mean_treatment*100:.3f}%)")
print(f"Difference (treatment − control): {difference*100:.4f} percentage points")
print(f"Welch t-statistic: {t_stat:.4f}")
print(f"Welch degrees of freedom: {welch_df:.2f}")
print(f"P-value: {t_pvalue:.6f}")
print(f"95% CI for difference: [{ci_low*100:.4f}, {ci_high*100:.4f}] percentage points")
print("Decision:", "Reject H0" if t_pvalue < ALPHA else "Fail to reject H0")
print("Interpretation:", 
      "There is statistically significant evidence of a difference in mean conversion outcomes."
      if t_pvalue < ALPHA else
      "The data do not provide sufficient evidence of a difference in mean conversion outcomes.")

## 7. Hypothesis Test 2 — Chi-square test of independence

**H₀:** Experimental group and conversion outcome are independent.  
**H₁:** Experimental group and conversion outcome are associated.

A 2 × 2 contingency table compares observed counts. The expected counts are also displayed.

In [ ]:
contingency = pd.crosstab(merged["group"], merged["converted"]).reindex(
    index=["control", "treatment"], columns=[0, 1], fill_value=0
)
contingency.index.name = "group"
contingency.columns = ["not_converted_0", "converted_1"]
display(contingency)

chi2_stat, chi2_pvalue, chi2_dof, expected = stats.chi2_contingency(contingency.to_numpy())
expected_df = pd.DataFrame(expected, index=contingency.index, columns=contingency.columns)

print(f"Chi-square statistic: {chi2_stat:.4f}")
print(f"Degrees of freedom: {chi2_dof}")
print(f"P-value: {chi2_pvalue:.6f}")
print("Expected frequencies:")
display(expected_df.round(3))
print("Decision:", "Reject H0" if chi2_pvalue < ALPHA else "Fail to reject H0")
print("Interpretation:",
      "There is evidence of an association between experimental group and conversion."
      if chi2_pvalue < ALPHA else
      "The data do not provide sufficient evidence of an association between experimental group and conversion.")

## 8. Hypothesis Test 3 — One-way ANOVA

### Analysis A: Conversion across countries

**H₀:** Mean conversion outcomes are equal across all countries.  
**H₁:** At least one country's mean conversion outcome differs.

This follows the report's one-way ANOVA approach. The test uses the actual country categories in the merged dataset.

In [ ]:
country_summary = (
    merged.groupby("country", observed=True)
    .agg(users=("user_id", "nunique"),
         conversions=("converted", "sum"),
         conversion_rate=("converted", "mean"))
    .sort_index()
)
country_summary["conversion_rate_percent"] = country_summary["conversion_rate"] * 100
display(country_summary)

country_samples = [
    group["converted"].astype(float).to_numpy()
    for _, group in merged.groupby("country", observed=True)
    if len(group) > 0
]
if len(country_samples) < 2:
    raise ValueError("Country ANOVA requires at least two non-empty country groups.")

country_f, country_pvalue = stats.f_oneway(*country_samples)
country_df_between = len(country_samples) - 1
country_df_within = len(merged) - len(country_samples)

print(f"ANOVA F-statistic: {country_f:.4f}")
print(f"Degrees of freedom: ({country_df_between}, {country_df_within})")
print(f"P-value: {country_pvalue:.6f}")
print("Decision:", "Reject H0" if country_pvalue < ALPHA else "Fail to reject H0")
print("Interpretation:",
      "At least one country has a different mean conversion outcome."
      if country_pvalue < ALPHA else
      "There is insufficient evidence that mean conversion outcomes differ across countries.")

### Analysis B: Country × experimental group

Create a category for each observed country–strategy combination, then run a one-way ANOVA across those categories.

**H₀:** Mean conversion outcomes are equal across all observed country–strategy combinations.  
**H₁:** At least one combination has a different mean conversion outcome.

**Limitation:** This one-way ANOVA does not formally test a country × treatment interaction. A factorial model would be needed for a formal interaction test.

In [ ]:
merged["country_strategy"] = (
    merged["group"].astype(str).str.title() + " – " + merged["country"].astype(str)
)
combo_summary = (
    merged.groupby("country_strategy", observed=True)
    .agg(users=("user_id", "nunique"),
         conversions=("converted", "sum"),
         conversion_rate=("converted", "mean"))
    .sort_index()
)
combo_summary["conversion_rate_percent"] = combo_summary["conversion_rate"] * 100
display(combo_summary)

combo_samples = [
    group["converted"].astype(float).to_numpy()
    for _, group in merged.groupby("country_strategy", observed=True)
    if len(group) > 0
]
if len(combo_samples) < 2:
    raise ValueError("Country–strategy ANOVA requires at least two non-empty combinations.")

combo_f, combo_pvalue = stats.f_oneway(*combo_samples)
combo_df_between = len(combo_samples) - 1
combo_df_within = len(merged) - len(combo_samples)

print(f"ANOVA F-statistic: {combo_f:.4f}")
print(f"Degrees of freedom: ({combo_df_between}, {combo_df_within})")
print(f"P-value: {combo_pvalue:.6f}")
print("Decision:", "Reject H0" if combo_pvalue < ALPHA else "Fail to reject H0")
print("Interpretation:",
      "At least one country–strategy combination has a different mean conversion outcome."
      if combo_pvalue < ALPHA else
      "There is insufficient evidence that mean conversion differs across country–strategy combinations.")

plot_data = combo_summary["conversion_rate_percent"]
ax = plot_data.plot(kind="bar", rot=35)
ax.set_title("Conversion rate by country–strategy combination")
ax.set_xlabel("Country–strategy combination")
ax.set_ylabel("Conversion rate (%)")
plt.tight_layout()
plt.show()

## 9. Results summary

The table below consolidates all four tests. For the ANOVA tests, degrees of freedom are reported as (between-groups, within-groups).

In [ ]:
def decision_text(p):
    return "Reject H0" if p < ALPHA else "Fail to reject H0"

results_summary = pd.DataFrame([
    {
        "Test": "Welch independent-samples t-test",
        "Research question": "Do mean conversion outcomes differ between control and treatment?",
        "Test statistic": f"t = {t_stat:.4f}",
        "Degrees of freedom": f"{welch_df:.2f}",
        "P-value": t_pvalue,
        "Alpha": ALPHA,
        "Decision": decision_text(t_pvalue),
        "Conclusion": "Difference detected" if t_pvalue < ALPHA else "No statistically significant difference detected"
    },
    {
        "Test": "Chi-square test of independence",
        "Research question": "Are experimental group and conversion outcome associated?",
        "Test statistic": f"chi-square = {chi2_stat:.4f}",
        "Degrees of freedom": str(chi2_dof),
        "P-value": chi2_pvalue,
        "Alpha": ALPHA,
        "Decision": decision_text(chi2_pvalue),
        "Conclusion": "Association detected" if chi2_pvalue < ALPHA else "No statistically significant association detected"
    },
    {
        "Test": "One-way ANOVA: country",
        "Research question": "Does mean conversion outcome differ across countries?",
        "Test statistic": f"F = {country_f:.4f}",
        "Degrees of freedom": f"({country_df_between}, {country_df_within})",
        "P-value": country_pvalue,
        "Alpha": ALPHA,
        "Decision": decision_text(country_pvalue),
        "Conclusion": "Country differences detected" if country_pvalue < ALPHA else "No statistically significant country differences detected"
    },
    {
        "Test": "One-way ANOVA: country–strategy",
        "Research question": "Does mean conversion differ across country–strategy combinations?",
        "Test statistic": f"F = {combo_f:.4f}",
        "Degrees of freedom": f"({combo_df_between}, {combo_df_within})",
        "P-value": combo_pvalue,
        "Alpha": ALPHA,
        "Decision": decision_text(combo_pvalue),
        "Conclusion": "Combination differences detected" if combo_pvalue < ALPHA else "No statistically significant combination differences detected"
    }
])
results_summary["P-value"] = results_summary["P-value"].map(lambda x: f"{x:.6f}")
display(results_summary)

## 10. Final conclusion

Use the results printed above, rather than assuming they match the reference report. A p-value ≥ 0.05 means **fail to reject** the null hypothesis; it does not prove that the pages or groups are identical.

- If the treatment conversion rate is higher but the relevant p-value is ≥ 0.05, the observed lift is not statistically significant at α = 0.05.
- If the treatment p-value is < 0.05, there is statistical evidence of a difference; inspect the direction and size of the rate difference before recommending a change.
- Country ANOVA tests whether any country means differ; it does not by itself establish which country differs.
- The six-combination ANOVA is not a formal interaction test.

If results differ from the reference report, compare the observed row counts, mismatch handling, duplicate-user handling, country merge matches, and actual category labels. Do not alter calculations just to force a match.

## 11. Export the cleaned data and reproducibility note

This optional cell saves the merged analysis dataset and summary table in the current working directory. These files are generated from the loaded datasets.

In [ ]:
merged.to_csv("ab_test_cleaned_merged.csv", index=False)
results_summary.to_csv("ab_test_results_summary.csv", index=False)
print("Saved: ab_test_cleaned_merged.csv")
print("Saved: ab_test_results_summary.csv")

## Quick explanation for your professor

1. **Welch t-test:** compares average 0/1 conversion outcomes between control and treatment; the mean of a binary variable is its conversion rate.
2. **Chi-square test:** checks whether conversion status is associated with which experimental group a visitor belongs to.
3. **Country ANOVA:** checks whether mean conversion outcomes differ among countries.
4. **Country–strategy ANOVA:** checks differences across observed country and strategy combinations, but is not a formal interaction test.

For every test, compare the p-value with α = 0.05. Reject H₀ if p < 0.05; otherwise, fail to reject H₀. “Not significant” does not mean “proven identical.”